# Lab 18-02 · A Bare-Metal Cortex-M4 Project, Built and Run

Six parts, each built on `bmbuild.py` (clang, ld.lld, the LLVM tools and QEMU). **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bmbuild as bb
OUT = Path(tempfile.mkdtemp(prefix="lab1802_"))
elf = bb.build(OUT / "fw.elf", mapfile=OUT / "fw.map")
print("built", elf.name)

built fw.elf


## Part 1 · Sections and symbols

*Predict:* the section and address of `magic`, `offsets`, `zeros`, `squares` and `fake_reg`.

In [2]:
for name, (size, addr) in bb.sections(elf).items():
    print(f"{name:12s} {size:6d} bytes at 0x{addr:08X}")
sym = bb.symbols(elf)
for s in ("magic", "offsets", "fake_reg", "zeros", "squares", "_etext", "_sidata", "_sdata", "_edata", "_sbss", "_ebss", "_estack"):
    print(f"{s:10s} 0x{sym[s]:08X}")
print("Berkeley size report:", bb.berkeley(elf))

.isr_vector      64 bytes at 0x00000000
.text          1452 bytes at 0x00000040
.ARM.exidx       24 bytes at 0x000005EC
.data            16 bytes at 0x20000000
.bss            260 bytes at 0x20000010
magic      0x20000000
offsets    0x20000004
fake_reg   0x2000000C
zeros      0x20000010
squares    0x000005E2
_etext     0x000005EC
_sidata    0x00000604
_sdata     0x20000000
_edata     0x20000010
_sbss      0x20000010
_ebss      0x20000114
_estack    0x20010000
Berkeley size report: {'text': 1540, 'data': 16, 'bss': 260}


## Part 2 · The vector table

*Predict:* words 0 and 1 of the image, and which function word 1 points to.

In [3]:
vt = bb.vector_table(elf)
print(f"word 0 (initial SP) = 0x{vt[0]:08X}   _estack = 0x{sym['_estack']:08X}")
print(f"word 1 (reset)      = 0x{vt[1]:08X}   Reset_Handler = 0x{sym['Reset_Handler']:08X}  (+1 = Thumb bit)")
print("words 2-15:", [f"0x{w:X}" for w in vt[2:]])

word 0 (initial SP) = 0x20010000   _estack = 0x20010000
word 1 (reset)      = 0x0000005D   Reset_Handler = 0x0000005C  (+1 = Thumb bit)
words 2-15: ['0x41', '0x41', '0x41', '0x41', '0x41', '0x0', '0x0', '0x0', '0x0', '0x41', '0x0', '0x0', '0x41', '0x41']


## Part 3 · Run it, then break it

*Predict:* the output and exit status with the full startup code, without the `.data` copy, and without the `.bss` zeroing.

In [4]:
for label, defs in (("full startup", ()), ("no .data copy", ("LAB_SKIP_DATA_COPY",)), ("no .bss zeroing", ("LAB_SKIP_BSS_ZERO",))):
    status, out = bb.run_qemu(bb.build(OUT / f"{len(defs)}{''.join(defs)}.elf", defines=defs))
    print(f"--- {label}: exit status {status}")
    print(out.strip())

--- full startup: exit status 0
TEST PASSED
--- no .data copy: exit status 1
FAIL: .data initial value
FAIL: .data array
FAIL: .data writable
FAIL: field insert bits [11:8]
FAIL: field clear bits [31:28]
FAIL: field extract bits [27:16]


--- no .bss zeroing: exit status 0
TEST PASSED


## Part 4 · `volatile`

*Predict:* the code of `wait_plain` and `wait_volatile` at `-O0`, `-O2` and `-Os`.

In [5]:
for opt in ("-O0", "-O2", "-Os"):
    obj = bb.compile_object("poll.c", OUT / f"poll{opt}.o", opt=opt)
    for fn in ("wait_plain", "wait_volatile"):
        ins = bb.disassemble(obj, fn)
        loads = sum(1 for i in ins if i.startswith("ldr"))
        print(f"{opt} {fn:14s} {len(ins):2d} instructions, {loads} loads: " + "; ".join(i.split("<")[0].strip() for i in ins))

-O0 wait_plain      8 instructions, 1 loads: b	0x2; movw	r0, #0x0; movt	r0, #0x0; ldr	r0, [r0]; cbnz	r0, 0x12; b	0x10; b	0x2; bx	lr
-O0 wait_volatile   8 instructions, 1 loads: b	0x16; movw	r0, #0x0; movt	r0, #0x0; ldr	r0, [r0]; cbnz	r0, 0x26; b	0x24; b	0x16; bx	lr
-O2 wait_plain      1 instructions, 0 loads: bx	lr
-O2 wait_volatile  15 instructions, 4 loads: movw	r0, #0x0; movt	r0, #0x0; b	0x1c; ldr	r1, [r0]; cmp	r1, #0x0; it	ne; bxne	lr; ldr	r1, [r0]; cbnz	r1, 0x26; ldr	r1, [r0]; cbnz	r1, 0x26; ldr	r1, [r0]; cmp	r1, #0x0; beq	0x10; bx	lr


-Os wait_plain      1 instructions, 0 loads: bx	lr
-Os wait_volatile   6 instructions, 1 loads: movw	r0, #0x0; movt	r0, #0x0; ldr	r1, [r0]; cmp	r1, #0x0; beq	0xc; bx	lr


## Part 5 · Optimization and size

*Predict:* the order of `.text` sizes from `-O0` to `-Oz`.

In [6]:
sizes = {}
for opt in ("-O0", "-O1", "-O2", "-Os", "-Oz"):
    e = bb.build(OUT / f"fw{opt}.elf", opt=opt)
    sizes[opt] = bb.sections(e)[".text"][0]
    status, _ = bb.run_qemu(e)
    print(f"{opt}: .text {sizes[opt]} bytes, QEMU exit {status}")
plt.bar(list(sizes), list(sizes.values()))
plt.ylabel(".text bytes"); plt.title("Lab firmware, clang, Cortex-M4"); plt.show()

-O0: .text 1504 bytes, QEMU exit 0


-O1: .text 1244 bytes, QEMU exit 0


-O2: .text 1452 bytes, QEMU exit 0


-Os: .text 1232 bytes, QEMU exit 0


-Oz: .text 860 bytes, QEMU exit 0


## Part 6 · The layout formulas

*Predict* `_sidata`, `_edata` and `_ebss` from the section sizes with §3.4's formulas.

In [7]:
s = bb.sections(elf)
sidata = sym["_etext"] + s.get(".ARM.exidx", (0, 0))[0]   # the unwind index sits between .text and .data's image
edata = bb.align(0x20000000 + s[".data"][0])
ebss = bb.align(edata + s[".bss"][0])
for name, pred in (("_sidata", sidata), ("_edata", edata), ("_ebss", ebss)):
    print(f"{name}: predicted 0x{pred:08X}, linker 0x{sym[name]:08X}, {'match' if pred == sym[name] else 'DIFFERENT'}")
# the map file shows each output section's run (VMA) and load (LMA) addresses
for line in open(OUT / "fw.map"):
    if line.split()[-1:] and line.split()[-1] in (".isr_vector", ".text", ".ARM.exidx", ".data", ".bss"):
        print(line.rstrip())

_sidata: predicted 0x00000604, linker 0x00000604, match
_edata: predicted 0x20000010, linker 0x20000010, match
_ebss: predicted 0x20000114, linker 0x20000114, match
       0        0       40     4 .isr_vector
      40       40      5ac     4 .text
     5ec      5ec       18     4 .ARM.exidx
20000000      604       10     4 .data
20000010 20000010      104     4 .bss


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).